# ANPR training on Colab A100 (skeleton)
Training only. Reproduce each run from: git commit + dataset hash + config yaml + seed. Fill `REPO_URL` first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, json, time, platform
DRIVE = '/content/drive/MyDrive/anpr'
REPO_URL = 'https://github.com/<user>/<repo>.git'   # TODO
RUN_ID = time.strftime('%Y%m%d_%H%M%S')
os.makedirs(f'{DRIVE}/runs/{RUN_ID}', exist_ok=True)

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
!git clone $REPO_URL /content/repo || (cd /content/repo && git pull)
%cd /content/repo
!pip install -q -r colab/requirements-colab.txt
!git rev-parse HEAD

In [ ]:
# dataset: unzip from Drive to local SSD (never train directly from Drive)
!mkdir -p /content/data && unzip -q -o $DRIVE/datasets/dataset_v2.zip -d /content/data
!cat /content/data/dataset_v2/version.json

In [ ]:
# environment record (mandatory per run)
import importlib.metadata as md
env = {'run_id': RUN_ID, 'python': platform.python_version(), 'torch': torch.__version__,
       'gpu': torch.cuda.get_device_name(0), 'commit': subprocess.getoutput('git rev-parse HEAD'),
       'packages': {d.metadata['Name']: d.version for d in md.distributions()}}
json.dump(env, open(f'{DRIVE}/runs/{RUN_ID}/env.json', 'w'), indent=1)

In [ ]:
# example: baseline training with resume. Hyper-parameters come from a yaml in configs/, not from here.
from ultralytics import YOLO
# model = YOLO('yolo11n.pt')
# model.train(data='/content/data/dataset_v2/anpr_detect.yaml', imgsz=640, epochs=100, batch=32, seed=42,
#             project=f'{DRIVE}/runs', name=RUN_ID, save_period=1, exist_ok=True)
# resume after disconnect: YOLO(f'{DRIVE}/runs/{RUN_ID}/weights/last.pt').train(resume=True)